# Gradient correctness of the reversible stacks

A reversible network never stores its intermediate activations — the backward pass *reconstructs*
them and re-runs each block. If that reconstruction or the re-run is subtly wrong, **the loss still
goes down** and nothing looks broken (§3 below is exactly that story). So correctness needs its own
tests, at two different levels:

| check | where | what it catches |
|---|---|---|
| `gradcheck.py` | **fp32, CPU, at init** | algebra: is the custom `autograd.Function` the same function as a store-everything reference? |
| `revcheck.py` | **GPU, bf16, trained weights** | conditioning + precision: does reconstruction still hold in the regime we actually train in? |

The second one exists because the first one **cannot** catch a class of bug that cost an entire round
of results (§3). Run order in this notebook: exact algebra (§1), the fix that makes it hold under
autocast (§2), the bug story (§3), then the numerical reality on GPU (§4).

In [1]:
import json, os, subprocess, sys, inspect
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "model.py").exists():
    ROOT = ROOT.parent          # notebook lives in notebooks/
assert (ROOT / "model.py").exists(), ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import torch
print("cwd         :", ROOT)
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("bf16 support:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "n/a")

cwd         : C:\Users\Monisha Srivastava\.zcode\workspace\default\reversible-20m
torch       : 2.10.0+cu126
cuda avail  : True | NVIDIA GeForce RTX 3060 Laptop GPU
bf16 support: True


## 1. Exact algebra — fp32 on CPU (`gradcheck.py`)

Tiny config (vocab 256, d_model 64, 4 blocks, 4 heads, seq 32, batch 2) so every gradient can be
compared against a reference that keeps all activations. Three things are checked:

- **`FusedCE`** — the chunked cross-entropy that keeps logits memory `O(chunk)` in both directions,
  against `F.cross_entropy(F.linear(h, W))`.
- **`rev_euler`** — the two-stream coupling `a += f(b); b += g(a)`, against the same recurrence run
  with plain autograd.
- **`rev_mid`** — leapfrog `x_{i+1} = x_{i-1} + f_i(x_i)`, likewise.

Both reversible models are loaded from the *baseline's* `state_dict`, so any difference is the
backward, not initialization. The assertion threshold is `max_rel_grad_err < 1e-5`.

In [2]:
import gradcheck as gc
from model import GPTRevEuler, GPTRevMid

gc.check_fused_ce()
e_err = gc.check_reversible(GPTRevEuler, "rev_euler")
m_err = gc.check_reversible(GPTRevMid, "rev_mid")
print(f"\nworst relative gradient error over every parameter: "
      f"rev_euler {e_err:.2e}, rev_mid {m_err:.2e}  (threshold 1e-5)")
print("ALL GRADIENT CHECKS PASSED")

[FusedCE] rel_loss_err=0.00e+00 dH_err=1.83e-06 dW_err=7.46e-07


[rev_euler] loss_ref=5.555954 loss_rev=5.555954 d_loss=0.00e+00  max_rel_grad_err=6.10e-07
[rev_mid] loss_ref=5.555785 loss_rev=5.555785 d_loss=0.00e+00  max_rel_grad_err=8.61e-07

worst relative gradient error over every parameter: rev_euler 6.10e-07, rev_mid 8.61e-07  (threshold 1e-5)
ALL GRADIENT CHECKS PASSED


## 2. What makes it hold under autocast

`gradcheck.py` runs in fp32 with no autocast. Training runs under `torch.autocast("cuda", bf16)`,
and that changes the rules: **autocast caches the bf16 copy of every weight**. The reversible
backward recomputes each block inside `torch.autograd.grad`, so it needs those casts to be
differentiable — which they are not if they were first produced inside `torch.no_grad()`.

Two pieces of `model.py` exist purely for this:

In [3]:
import model
print(inspect.getsource(model._amp_state))
print(inspect.getsource(model._amp_ctx))
print(inspect.getsource(model._require_all_grads))

# the whole point: the backward re-enters autocast with the weight cache DISABLED
st = (True, torch.bfloat16)
ctx = model._amp_ctx(st)
print("_amp_ctx cache_enabled:", ctx._cache_enabled, "  (must be False)")
assert ctx._cache_enabled is False

def _amp_state():
    return (torch.is_autocast_enabled("cuda"), torch.get_autocast_dtype("cuda"))

def _amp_ctx(state):
    enabled, dtype = state
    return torch.autocast("cuda", dtype=dtype, enabled=enabled, cache_enabled=False)

def _require_all_grads(grads, params):
    if any(g is None for g in grads):
        missing = sum(g is None for g in grads)
        raise RuntimeError(
            f"reversible backward produced no gradient for {missing}/{len(params)} block "
            "parameters; the autocast weight cache has detached them from the graph")

_amp_ctx cache_enabled: False   (must be False)


In [4]:
# where the fix is applied: every recompute site in the reversible backwards
src = (ROOT / "model.py").read_text().splitlines()
for i, line in enumerate(src, 1):
    if "_amp_ctx" in line or "_require_all_grads" in line:
        print(f"{i:4d}: {line}")

  43: def _amp_ctx(state):
  48: def _require_all_grads(grads, params):
 216:                 with _amp_ctx(ctx.amp), torch.no_grad():
 222:             with _amp_ctx(ctx.amp), torch.enable_grad():
 229:             _require_all_grads(grads[2:], params)
 273:                 with _amp_ctx(ctx.amp), torch.no_grad():
 278:             with _amp_ctx(ctx.amp), torch.enable_grad():
 284:             _require_all_grads(grads[2:], params)


## 3. The bug this catches, and why §1 could not see it

Under `torch.autocast`, the bf16 copy of each weight is **cached**. A reversible backward
recomputes blocks inside `torch.autograd.grad`, but the forward ran under `torch.no_grad()` — so the
cached bf16 copy carries **no `grad_fn`**, the recompute cannot reach the fp32 parameter, and
`torch.autograd.grad(..., allow_unused=True)` silently returns `None` for **every `Linear` weight in
every block**.

Consequence: all attention and MLP matrices were **frozen at their initialization**. Only the
LayerNorm gains and the embedding trained. The loss still fell from 7.5 to 4.9, so every dashboard
looked healthy — and **`gradcheck.py` passed at 6e-7 relative error the whole time**, because it runs
fp32 on CPU where there is no weight cache.

> **Correctness tests must run in the precision and on the device you actually train in.**

The tell was a **flat learning-rate response**: a model that barely cares about its LR over a 4x
range is usually not training at all.

In [5]:
# the superseded runs are kept in the repo under runs/buggy_* -- this is what "frozen" looks like
import re
RUNS = ROOT / "runs"


def metrics(name):
    p = RUNS / name / "metrics.json"
    return json.loads(p.read_text()) if p.exists() else None


print("BROKEN (block weights frozen): rev_mid, 10M tokens, batch 16")
for d in sorted(RUNS.glob("buggy_s2_rev_mid_*")):
    m = metrics(d.name)
    print(f"  lr {m['lr']:<8g} val {m['final_val_loss']:.4f}")
print("  -> 4x LR range, ~0.02 nats of spread. Nothing is learning.\n")

print("FIXED: same arch, same horizon")
for d in sorted(RUNS.glob("s3_rev_mid_0.*")):
    m = metrics(d.name)
    print(f"  lr {m['lr']:<8g} val {m['final_val_loss']:.4f}")
print("  -> sharp interior minimum at 7.5e-4, 0.18 nats of spread over 4x.")

print("\n`_require_all_grads` now raises instead of accepting a None gradient, so this exact failure")
print("cannot recur silently.")

BROKEN (block weights frozen): rev_mid, 10M tokens, batch 16
  lr 0.0015   val 5.3044
  lr 0.003    val 5.2830
  lr 0.006    val 5.2795
  -> 4x LR range, ~0.02 nats of spread. Nothing is learning.

FIXED: same arch, same horizon
  lr 0.0004   val 4.8526
  lr 0.00075  val 4.6716
  lr 0.0015   val 4.8455
  -> sharp interior minimum at 7.5e-4, 0.18 nats of spread over 4x.

`_require_all_grads` now raises instead of accepting a None gradient, so this exact failure
cannot recur silently.


## 4. The regime that actually matters — GPU, bf16, trained weights (`revcheck.py`)

`revcheck.py` forwards once keeping every activation, then inverts from the final pair exactly as the
reversible backward does, and compares — both the reconstructed activations and the resulting
parameter gradients — against a store-everything autograd reference **run under the same autocast**.
It reports per-block reconstruction error, the worst parameter gradient error, and the minimum cosine
similarity between the reversible gradient and the true gradient.

This is where reversibility stops being free: **`x_0` cannot be reconstructed.** It is
embedding-scale, but the backward recovers it by subtracting six blocks' worth of accumulated
residual — catastrophic cancellation, not rounding. Running in fp32 only lifts block-0 cosine
similarity to ~0.24: precision does not fix an ill-conditioned subtraction.

Two things to keep in mind when reading the output below:

- The committed `--steps 300` artifacts (trained weights) are the honest setting. `--steps 0` (at
  init, reproduced live further down) looks fine — `x_0` off by 6%, every cosine ≥0.9997 — and would
  have hidden the problem entirely.
- `revcheck.py` has since gained a labelled output schema. In the older committed files the
  reconstruction list is ordered from the input side outward, and its **first** entry is leapfrog's
  `x_{-1}`, which is identically zero — a relative error against zero is meaningless, so the newer
  script reports `null` there. The cell labels both schemas correctly.

In [6]:
def recon_items(d):
    """Label the reconstruction-error entries. Both schemas are ordered from the input side
    outward, so entry 0 is the activation closest to the embedding. For leapfrog that is
    x_{-1}, the all-zero seed -- a relative error against zero is undefined, which the newer
    revcheck.py reports as null and the older one as a huge number. x_0 is the interesting one."""
    if "recon_rel_err_by_layer" in d:                  # newer revcheck.py: already labelled
        return list(d["recon_rel_err_by_layer"].items())
    vals = d["recon_rel_err_per_layer"]
    if d["arch"].startswith("rev_euler"):
        return [(f"(a,b) entering block {i}", v) for i, v in enumerate(vals)]
    return list(zip(["x_-1 (zero seed)"] + [f"x_{i}" for i in range(len(vals) - 1)], vals))


for p in sorted(RUNS.glob("revcheck_*.json")):
    txt = p.read_text()
    d = json.loads(txt[txt.index("{"):])
    print(f"=== {p.name}  ({d['arch']}, trained_steps={d['trained_steps']}, bf16={d['bf16']}) ===")
    print(f"  forward: reversible {d['loss_rev']:.6f} vs reference {d['loss_ref']:.6f}"
          f"  (delta {abs(d['loss_rev']-d['loss_ref']):.1e}) -- the forward is exact")
    print("  reconstruction relative error, embedding side first:")
    for name, e in recon_items(d):
        if e is None:
            print(f"      {name:26s} undefined (reference is all-zero)")
        else:
            print(f"      {name:26s} {e:.3e}" + ("   <-- unusable" if e > 1.0 else ""))
    print(f"  worst parameter    : {d['worst_param']}")
    print(f"  max grad rel err   : {d['max_grad_rel_err']:.3g}")
    print(f"  min grad cosine sim: {d['min_grad_cosine_sim']:.4f}\n")

print("Read it from the bottom up: x_1..x_4 come back to ~1e-4-1e-3, and then x_0 -- the very first")
print("activation -- fails outright (rel err 87 with trained weights, vs 0.065 at init while")
print("everything is still small). x_-1 is the zero seed, so its ratio is meaningless, not a finding.")
print("")
print("In the gradients the same effect appears as a collapse of cosine similarity: 0.15 minimum with")
print("trained weights vs 0.9997 at init, and REPORT.md section 5b's per-block breakdown localises it")
print("to block 0 (0.054) with blocks 1-5 at 0.998-1.000. rev_euler is milder (0.75) but has the same")
print("pathology at its first pair.")

=== revcheck_euler_trained.json  (rev_euler, trained_steps=300, bf16=True) ===
  forward: reversible 6.608437 vs reference 6.608437  (delta 0.0e+00) -- the forward is exact
  reconstruction relative error, embedding side first:
      (a,b) entering block 0     3.175e+03   <-- unusable
      (a,b) entering block 1     2.564e-03
      (a,b) entering block 2     1.786e-03
      (a,b) entering block 3     1.418e-03
      (a,b) entering block 4     6.064e-04
      (a,b) entering block 5     4.525e-04
  worst parameter    : blocks.0.attn.qkv.weight
  max grad rel err   : 1.9
  min grad cosine sim: 0.7543

=== revcheck_mid_init.json  (rev_mid, trained_steps=0, bf16=True) ===
  forward: reversible 9.092338 vs reference 9.092338  (delta 0.0e+00) -- the forward is exact
  reconstruction relative error, embedding side first:
      x_-1 (zero seed)           2.045e+10   <-- unusable
      x_0                        6.468e-02
      x_1                        1.215e-02
      x_2                     

In [7]:
# reproduce it live, at init (fast; needs data/train.bin from data_prep.py)
if torch.cuda.is_available() and (ROOT / "data" / "train.bin").exists():
    r = subprocess.run([sys.executable, "revcheck.py", "--arch", "rev_mid", "--steps", "0",
                        "--bf16", "1"], capture_output=True, text=True, cwd=ROOT, timeout=900)
    print(r.stdout.strip() or r.stderr[-2000:])
else:
    print("skipped: needs a CUDA GPU and data/train.bin (run `python data_prep.py` first).")
    print("The committed runs/revcheck_*.json above hold the results from the original hardware.")

no-grad params (reversible path): []
no-grad params (reference path): []
{
  "arch": "rev_mid",
  "trained_steps": 0,
  "bf16": true,
  "loss_rev": 9.092337608337402,
  "loss_ref": 9.092337608337402,
  "recon_rel_err_by_layer": {
    "x_-1": null,
    "x_0": 0.064677,
    "x_1": 0.012154,
    "x_2": 0.004143,
    "x_3": 0.002235,
    "x_4": 0.0
  },
  "max_recon_rel_err": 0.06467670202255249,
  "max_grad_rel_err": 0.018633540719747543,
  "worst_param": "blocks.0.mlp.fc.weight",
  "min_grad_cosine_sim": 0.9996968507766724,
  "grad_cosine_by_block": {
    "block0": 0.9997,
    "block1": 1.0,
    "block2": 1.0,
    "block3": 1.0,
    "block4": 1.0,
    "block5": 1.0
  }
}


### Fixing it: `rev_mid_seed`

`x_0` is already an *input* to the reversible `Function`, so it can simply be stored instead of
reconstructed — one extra activation, no algebra changes. That is the `rev_mid_seed` arch
(`exact_seed = True`):

| | block-0 cosine | max grad rel err | val @10M | peak alloc (b=16) | max batch |
|---|---|---|---|---|---|
| `rev_mid` | 0.054 | 1.20 | **4.6716** | 1157 MB | 192 |
| `rev_mid_seed` | **0.9999** | **0.031** | 4.6731 | 1176 MB | 176 |

**And it changes nothing.** Restoring block 0's gradient from near-random to exact moves val loss by
0.0015 nats — noise — while costing ~179 MB at batch 192, enough to drop the max batch below 192. At
6 layers the first block is doing little enough work that a garbage gradient there is harmless. I
would expect that to stop being true with depth, which is why the variant stays in the repo.

In [8]:
from model import ARCHS
print("architectures available in model.py:")
for k, v in ARCHS.items():
    print(f"  {k:14s} {v.__name__:16s} exact_seed={getattr(v, 'exact_seed', False)}")

pair = {}
for d in sorted(RUNS.glob("s3_rev_mid*0.00075")):
    m = metrics(d.name)
    pair[m["arch"]] = m
for k, m in pair.items():
    print(f"\n{k:14s} val {m['final_val_loss']:.4f}  peak alloc {m['peak_mem_allocated_mb']:.0f} MB"
          f"  tok/s {m['tokens_per_s']/1e3:.1f}k")
if len(pair) == 2:
    a, bb = pair["rev_mid"], pair["rev_mid_seed"]
    print(f"\ndelta: val {bb['final_val_loss']-a['final_val_loss']:+.4f} nats, "
          f"memory {bb['peak_mem_allocated_mb']-a['peak_mem_allocated_mb']:+.0f} MB at batch 16")

architectures available in model.py:
  baseline       GPTBaseline      exact_seed=False
  rev_euler      GPTRevEuler      exact_seed=False
  rev_mid        GPTRevMid        exact_seed=False
  rev_mid_seed   GPTRevMidSeed    exact_seed=True
  rev_euler_seed GPTRevEulerSeed  exact_seed=True

rev_mid        val 4.6716  peak alloc 1157 MB  tok/s 30.1k

rev_mid_seed   val 4.6731  peak alloc 1176 MB  tok/s 27.2k

delta: val +0.0015 nats, memory +19 MB at batch 16


## Summary

1. The custom reversible `Function`s are **algebraically exact** — fp32/CPU gradients match a
   store-everything reference to ~1e-7, well under the 1e-5 assertion.
2. Under bf16 autocast that is *not enough*: the weight cache silently detached every block
   parameter from the graph. The fix is re-entering autocast with `cache_enabled=False` in the
   backward, plus a hard failure (`_require_all_grads`) so it can never be silent again.
3. On GPU with trained weights, blocks near the output reconstruct to ~1e-4–1e-3. **Block 0 does
   not reconstruct at all** — it is an ill-conditioned subtraction, and fp32 does not rescue it.
4. Storing `x_0` (`rev_mid_seed`) fixes the gradient exactly and changes the loss by nothing at 6
   layers — so the cheap variant is the one used for the headline runs.

Full write-up: [`REPORT.md`](../REPORT.md) · results: [`02_results_and_analysis.ipynb`](02_results_and_analysis.ipynb)